# GeoTIFF to Zarr

This notebook demonstrates how to convert GeoTIFF(s) to a Zarr store.

First, we import the required packages.

In [1]:
import xarray as xr
import rioxarray
import pystac_client
import rasterio
import zarr
import icechunk
import datetime
import numpy as np

from tqdm import tqdm
from pathlib import Path
from typing import List, Dict

import warnings
warnings.filterwarnings("ignore")

Then we set up the local input and output directories for the GeoTIFF and Zarr data.

In [2]:
gt_dirpath = Path('../data/02_cgls_ssm_geotiff')
cgls_dirpath = Path('../outputs/02_cgls_ssm_zarr')

gt_dirpath.mkdir(parents=True, exist_ok=True)
cgls_dirpath.mkdir(parents=True, exist_ok=True)

Next, we define a helper function to query eodc's STAC API for items of the Copernicus Global Land Service Soil Moisture collection (`CGLS_SSM_1KM`) intersecting a bounding box and date range.

In [3]:
def collect_cgls_ssm(bbox: List[float],
                    daterange: str,
                    num_items: int | None = None,) -> Dict[str, datetime.datetime]:
    
    stac_url = "https://stac.eodc.eu/api/v1/"
    collection_id = "CGLS_SSM_1KM"

    eodc = pystac_client.Client.open(stac_url)
    found = eodc.search(
        collections=[collection_id],
        bbox=bbox,
        datetime=daterange,
        max_items=num_items
    )

    meta = {}
    for item in tqdm(found.items()):
        href = item.assets['SSM'].href
        meta[href] = item.datetime

    return meta

Now we use this function to collect up to 100 SSM GeoTIFF files covering central Europe.

In [4]:
cgls_ssm_meta = collect_cgls_ssm(bbox=[7.,46.,10.,50.],
                                daterange="2018-07-01/2026-07-31",
                                num_items=100)
cgls_ssm_meta

100it [00:00, 100.12it/s]


{'https://objectstore.eodc.eu:2222/swift/v1/AUTH_68e13833a1624f43ba2cac01376a18af/cgls_ssm_1km/E048N018T6/c_gls_SSM1km_202204040000_CEURO_S1CSAR_V1.1.1_EU1K0M_E048N018T6.tif': datetime.datetime(2022, 4, 4, 0, 0, tzinfo=tzutc()),
 'https://objectstore.eodc.eu:2222/swift/v1/AUTH_68e13833a1624f43ba2cac01376a18af/cgls_ssm_1km/E048N012T6/c_gls_SSM1km_202204040000_CEURO_S1CSAR_V1.1.1_EU1K0M_E048N012T6.tif': datetime.datetime(2022, 4, 4, 0, 0, tzinfo=tzutc()),
 'https://objectstore.eodc.eu:2222/swift/v1/AUTH_68e13833a1624f43ba2cac01376a18af/cgls_ssm_1km/E042N018T6/c_gls_SSM1km_202204040000_CEURO_S1CSAR_V1.1.1_EU1K0M_E042N018T6.tif': datetime.datetime(2022, 4, 4, 0, 0, tzinfo=tzutc()),
 'https://objectstore.eodc.eu:2222/swift/v1/AUTH_68e13833a1624f43ba2cac01376a18af/cgls_ssm_1km/E048N012T6/c_gls_SSM1km_202203200000_CEURO_S1CSAR_V1.1.1_EU1K0M_E048N012T6.tif': datetime.datetime(2022, 3, 20, 0, 0, tzinfo=tzutc()),
 'https://objectstore.eodc.eu:2222/swift/v1/AUTH_68e13833a1624f43ba2cac01376a18af/c

## Conversion

With the GeoTIFF URI's now available, we can stack them into a single `xarray` dataset along the time dimension representing a datacube. This datacube is the basis for all the Zarr writing below.

In [5]:
dars = []
for href in cgls_ssm_meta.keys():
    da = rioxarray.open_rasterio(href, mask_and_scale=False)
    da = da[0].expand_dims(time=[cgls_ssm_meta[href].replace(tzinfo=None)])
    dars.append(da)

cube_dar = xr.concat(dars, dim="time", fill_value=255)
cube_dar

<xarray.DataArray (time: 100, y: 1200, x: 1200)> Size: 144MB
array([[[255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        ...,
        [255, 255, 255, ..., 251, 251, 251],
        [255, 255, 255, ..., 251, 251, 251],
        [255, 255, 255, ..., 251, 251, 251]],

       [[255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        ...,
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255]],

       [[255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        ...,
...
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255]],

       [[255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        ...,
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255]],

       [[255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        ...,
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255],
        [255, 255, 255, ..., 255, 255, 255]]],
      shape=(100, 1200, 1200), dtype=uint8)
Coordinates:
  * time         (time) datetime64[us] 800B 2022-04-04 2022-04-04 ... 2021-06-14
  * y            (y) float64 10kB 1.2e+06 1.202e+06 ... 2.398e+06 2.4e+06
  * x            (x) float64 10kB 4.2e+06 4.202e+06 ... 5.398e+06 5.4e+06
    band         int64 8B 1
    spatial_ref  int64 8B 0
Attributes:
    AREA_OR_POINT:  Area
    _FillValue:     255
    scale_factor:   1.0
    add_offset:     0.0

Once we have created the datacube, we can write to a file. Writing it to a COG is not (really) possible, as too many bands overwhelm the GDAL processes behind `rasterio`. Thus, users are limited in writing multidimensional data to a COG, whereas the Zarr format natively supports multidimensional arrays. 

Below is an example for aggregating a subset of the datacube over time and convert it to a COG.

In [6]:
sub_cube_dar = cube_dar.sel(y=slice(1840500, 1850500), x=slice(4970500, 4980500))
mean_dar = sub_cube_dar.mean(dim=["time"], skipna=True)
mean_dar.rio.to_raster(
        cgls_dirpath / 'subcube_mean_over_time.tif',
        compress="ZSTD",
        tiled=True,
        blockxsize=512,
        blockysize=512,
        driver='COG',
        dtype=mean_dar.dtype,
    )

There are several options for writing Zarr stores. The most user-friendly and straightforward method is to use `xarray`. 

In [7]:
SSM_VAR_NAME = "ssm"
SSM_DTYPE = "uint8"
SSM_FILL_VALUE = 255
SSM_CHUNKS = (16, 256, 256)

cube_ds = xr.Dataset({SSM_VAR_NAME: cube_dar.rio.write_nodata(SSM_FILL_VALUE)})

# add CF/geospatial metadata via rioxarray
cube_ds = cube_ds.rio.write_crs(cube_dar.rio.crs)
cube_ds = cube_ds.rio.write_coordinate_system()
cube_ds = cube_ds.rio.write_transform()

zarr_path = cgls_dirpath / 'xarray_cube.zarr'

# define the chunking along time x width x height
encoding = {SSM_VAR_NAME: {"chunks": SSM_CHUNKS, "dtype": SSM_DTYPE}}
cube_ds.to_zarr(zarr_path, mode='w', zarr_format=3, compute=False, encoding=encoding)
zarr.consolidate_metadata(zarr_path)

<Group file://../outputs/02_cgls_ssm_zarr/xarray_cube.zarr>

As an alternative and for more granular access, the same datacube may be written using the native `zarr` library.

In [8]:
native_zarr_path = cgls_dirpath / 'zarr_cube.zarr'

root = zarr.open_group(
    native_zarr_path,
    mode="w",
    zarr_format=3,
)

arr = root.create_array(
    "x",
    shape=cube_dar.x.shape,
    dtype=cube_dar.x.dtype,
    chunks=(SSM_CHUNKS[2],),
    dimension_names=("x",),
)
root["x"][:] = cube_dar.x.values

root.create_array(
    "y",
    shape=cube_dar.y.shape,
    dtype=cube_dar.y.dtype,
    chunks=(SSM_CHUNKS[1],),
    dimension_names=("y",),
)
root["y"][:] = cube_dar.y.values

root.create_array(
    "time",
    shape=cube_dar.time.shape,
    dtype=cube_dar.time.dtype,
    chunks=(SSM_CHUNKS[0],),
    dimension_names=("time",),
)
root["time"][:] = cube_dar.time.values

root.create_array(
    SSM_VAR_NAME,
    shape=cube_dar.shape,
    dtype=cube_dar.dtype,
    chunks=SSM_CHUNKS,
    fill_value=SSM_FILL_VALUE,
    dimension_names=("time", "y", "x"),
)
root[SSM_VAR_NAME][:] = cube_dar.values

zarr.consolidate_metadata(native_zarr_path)

<Group file://../outputs/02_cgls_ssm_zarr/zarr_cube.zarr>

## Virtualization

Instead of reprocessing or conversion (which consumes resources and may not be necessary for all applications) we can also virtualize the data repository. Virtualizing works by saving only the metadata of files into a Zarr array (made possible by _Icechunk_ or _Kerchunk_) with the only limitation being the original size of input file coordinates, as these will have to be loaded into memory when creating the virtual data representation. 

For virtualizing our sample dataset, we need to import more packages:

In [9]:
from virtualizarr import open_virtual_dataset
from virtual_tiff import VirtualTIFF
from obstore.store import HTTPStore
from obspec_utils.registry import ObjectStoreRegistry
from imagecodecs.zarr import register_codecs
register_codecs()

In this sample the downloaded SM data will not be aggregated into a single datacube, as concatenating the image coordinates into a single array would require loading the full arrays into memory and processing them to align their respective spatial positioning. Instead we will structure them according to the Equi7Grid tiling scheme (in which the data is already published). This reduces processign requirements and allows the virtualization process. If a single output store is wanted, the data has to be reprocessed as shown earlier such a pipeline is currently not supported by the `virtualizarr` API (reported in https://github.com/virtual-zarr/virtual-tiff/issues/55).

In [10]:
bucket_name = "cgls_ssm_1km"
ref_url = list(cgls_ssm_meta.keys())[0]
root_url = ref_url.split(bucket_name)[0] + bucket_name # https://objectstore.eodc.eu:2222/swift/v1/AUTH_68e13833a1624f43ba2cac01376a18af/cgls_ssm_1km
store = HTTPStore.from_url(root_url)
registry = ObjectStoreRegistry({root_url: store})

# read in tile metadata for tile-dependent processing
tiled_urls = {}
for url in cgls_ssm_meta.keys():
    tile_id = Path(url).stem.split('_')[-1][:-2]
    if tile_id in tiled_urls.keys():
        tiled_urls[tile_id].append(url)
    else:
        tiled_urls[tile_id] = [url]

For each Equi7Grid tile, we build a virtual dataset by reading only the metadata (not the pixel data) of each GeoTIFF via `virtualizarr`, attach coordinates and CRS, concatenate along time, and commit the result to a local Icechunk store.

In [ ]:
for tile_id, urls in tiled_urls.items():
    vzarr_outpath = cgls_dirpath / f'virtual_tile_{tile_id}'

    # gather the metadata of each image
    vdss = []
    for url in tqdm(urls, desc=f'Processing: {tile_id}'):
    
        # load virtual array -> not into memory except the metadata
        vcube_ds = open_virtual_dataset(
            url=url,
            registry=registry,
            parser=VirtualTIFF(ifd=0),
            loadable_variables=['x', 'y']
        )

        vcube_ds = vcube_ds.rename({'0': SSM_VAR_NAME})
        vcube_ds = vcube_ds.expand_dims(time=[cgls_ssm_meta[url].replace(tzinfo=None)])

        # get coordinates and CRS info
        dar = rioxarray.open_rasterio(url)
        vcube_ds = vcube_ds.assign_coords(
            x=("x", dar.x.data),
            y=("y", dar.y.data),
        )
        vcube_ds = vcube_ds.rio.write_crs(dar.rio.crs)
        vdss.append(vcube_ds)

    # concatenating across tiles is not possible as it requires fancy indexing -> rearrange by tiles instead
    vcube_ds = xr.concat(vdss, 
                    dim="time", 
                    fill_value=SSM_FILL_VALUE, 
                    join='outer')
    
    # writing to icechunk
    # remote HTTP object store containing the original tif files, referenced by root_url.
    # icechunk's VirtualChunkContainer requires the url_prefix to end in '/', unlike the
    # ObjectStoreRegistry above, so it gets its own slash-terminated variant here.
    vcc_url_prefix = root_url + "/"
    config = icechunk.RepositoryConfig.default()
    http_storage = icechunk.ObjectStoreConfig.Http(None, None)
    config.set_virtual_chunk_container(icechunk.VirtualChunkContainer(url_prefix=vcc_url_prefix, 
                                                                    store=http_storage))
    
    # define directory for output directory
    storage = icechunk.local_filesystem_storage(str(vzarr_outpath))
    repo = icechunk.Repository.create(storage=storage,
                                    config=config,
                                    authorize_virtual_chunk_access={vcc_url_prefix: icechunk.credentials.HttpAccess})
    session = repo.writable_session('main')
    
    # actually write data
    vcube_ds.vz.to_icechunk(session.store)
    snapshot = session.commit("Initial virtual cube")

Processing: E048N018: 100%|██████████| 24/24 [00:03<00:00,  7.01it/s]
  2026-07-29T10:11:31.568978Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324

Processing: E048N012: 100%|██████████| 26/26 [00:03<00:00,  7.12it/s]
  2026-07-29T10:11:35.376566Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324

Processing: E042N018: 100%|██████████| 25/25 [00:02<00:00,  8.59it/s]
  2026-07-29T10:11:38.380909Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores

We can now open one of the virtual Icechunk stores directly with `xarray`, just like a regular Zarr store.

In [12]:
repo_path = cgls_dirpath / 'virtual_tile_E048N012'
storage = icechunk.local_filesystem_storage(str(repo_path))
repo = icechunk.Repository.open(storage=storage)

session = repo.readonly_session("main")
store = session.store
vcube_ds = xr.open_zarr(store)
vcube_ds

  2026-07-29T10:11:40.871042Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:324



<xarray.Dataset> Size: 37MB
Dimensions:      (time: 26, y: 600, x: 600)
Coordinates:
  * time         (time) datetime64[ns] 208B 2022-04-04 2022-03-20 ... 2021-06-14
  * y            (y) float64 5kB 1.8e+06 1.798e+06 ... 1.202e+06 1.2e+06
  * x            (x) float64 5kB 4.8e+06 4.802e+06 ... 5.398e+06 5.4e+06
    spatial_ref  int64 8B ...
Data variables:
    ssm          (time, y, x) float32 37MB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>

Finally, we confirm the virtual store returns the same values as the native Zarr store for the same spatial subset and temporal aggregation.

In [13]:
cube_ds = xr.open_dataset(native_zarr_path)
cube_ds = cube_ds.sel(y=slice(1400500, 1500500), x=slice(5070500, 5200500)).mean(dim=["time"], skipna=True)

vcube_ds.sel(y=slice(1400500, 1500500), x=slice(5070500, 5200500)).mean(dim=["time"], skipna=True)
arrs_same = bool((cube_ds == vcube_ds).all())

print(f'Array values match: {arrs_same}')

Array values match: True
